# Financial News Sentiment Analysis

**데이터셋**: FinancialPhraseBank (Malo et al., 2014)  
**출처**: Kaggle - Sentiment Analysis for Financial News  
**목표**: 금융 뉴스 문장을 Positive / Neutral / Negative 3개 클래스로 분류

**모델 비교**: MLP → CNN → LSTM → GRU

## 1. 라이브러리 임포트

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Dense, Embedding, Flatten, Dropout,
                                      Conv1D, GlobalMaxPooling1D, GRU, LSTM)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print(f'TensorFlow: {tf.__version__}')

I0000 00:00:1775544367.627510    1703 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1775544368.310498    1703 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775544370.437001    1703 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow: 2.21.0


## 2. 데이터 로드 및 탐색

In [2]:
df = pd.read_csv('./data/all-data.csv', header=None, names=['label', 'sentence'], encoding='latin-1')

print(f'데이터 크기: {df.shape}')
print('\n클래스 분포:')
print(df['label'].value_counts())
print('\n결측값:')
print(df.isnull().sum())
df['length'] = df['sentence'].apply(len)
print('\n문장 길이 통계:')
print(df['length'].describe())

데이터 크기: (4846, 2)

클래스 분포:
label
neutral     2879
positive    1363
negative     604
Name: count, dtype: int64

결측값:
label       0
sentence    0
dtype: int64

문장 길이 통계:
count    4846.000000
mean      128.132068
std        56.526180
min         9.000000
25%        84.000000
50%       119.000000
75%       163.000000
max       315.000000
Name: length, dtype: float64


## 3. 데이터 분리 (Train / Test)

In [3]:
X = df['sentence']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {len(X_train)}, Test: {len(X_test)}')
print('\nTrain 클래스 분포:')
print(y_train.value_counts())

Train: 3876, Test: 970

Train 클래스 분포:
label
neutral     2303
positive    1090
negative     483
Name: count, dtype: int64


## 4. 전처리

### 4-1. 토크나이징 & 패딩

- `padding='post'`: 문장 뒤에 0을 채움
- 금융 뉴스는 문장 뒷부분(but, however 이후)이 감성을 뒤집는 경우가 많아
  `pre` 패딩이 LSTM/GRU에 유리할 수 있으나, MLP/CNN은 순서를 고려하지 않아 차이 없음

In [4]:
MAX_WORDS = 5000
MAX_LEN = 100
EMBED_DIM = 32

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')

print(f'X_train_pad: {X_train_pad.shape}')
print(f'X_test_pad:  {X_test_pad.shape}')

X_train_pad: (3876, 100)
X_test_pad:  (970, 100)


### 4-2. 라벨 인코딩 (원-핫 인코딩)

In [5]:
le = LabelEncoder()
y_train_enc = to_categorical(le.fit_transform(y_train), num_classes=3)
y_test_enc = to_categorical(le.transform(y_test), num_classes=3)

print(f'클래스 순서: {le.classes_}')
print(f'  인덱스 0 = {le.classes_[0]}')
print(f'  인덱스 1 = {le.classes_[1]}')
print(f'  인덱스 2 = {le.classes_[2]}')

클래스 순서: ['negative' 'neutral' 'positive']
  인덱스 0 = negative
  인덱스 1 = neutral
  인덱스 2 = positive


## 5. 모델 학습 및 비교

공통 조건:
- EarlyStopping: `patience=5`, `monitor='val_loss'`, `restore_best_weights=True`
- 각 모델마다 EarlyStopping 객체를 새로 생성 (내부 카운터 초기화 문제 방지)
- epochs=50 (EarlyStopping이 자동으로 최적 시점에서 종료)

### 5-1. MLP

In [6]:
early_stop_mlp = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

mlp_model = Sequential([
    Embedding(MAX_WORDS, EMBED_DIM),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

mlp_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history_mlp = mlp_model.fit(
    X_train_pad, y_train_enc,
    epochs=50, batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop_mlp],
    verbose=1
)

Epoch 1/50


I0000 00:00:1775544373.036194    1703 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5563 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9
I0000 00:00:1775544374.173655    1840 service.cc:153] XLA service 0x74b2bc042c60 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775544374.173681    1840 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9 (Driver: 12.7.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.20.0)
I0000 00:00:1775544374.218724    1840 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1775544374.416480    1840 cuda_dnn.cc:461] Loaded cuDNN version 92000
I0000 00:00:1775544374.446227    1840 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1794__.20


75/97 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5868 - loss: 0.9585

I0000 00:00:1775544376.645348    1840 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
I0000 00:00:1775544376.969379    1839 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1794__.20


97/97 ━━━━━━━━━━━━━━━━━━━━ 7s 40ms/step - accuracy: 0.5897 - loss: 0.9306 - val_accuracy: 0.6070 - val_loss: 0.8583
Epoch 2/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6426 - loss: 0.7838 - val_accuracy: 0.6778 - val_loss: 0.7499
Epoch 3/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7771 - loss: 0.5285 - val_accuracy: 0.6843 - val_loss: 0.7334
Epoch 4/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8571 - loss: 0.3157 - val_accuracy: 0.7023 - val_loss: 0.8321
Epoch 5/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9381 - loss: 0.1653 - val_accuracy: 0.7088 - val_loss: 0.9910
Epoch 6/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9877 - loss: 0.0529 - val_accuracy: 0.7139 - val_loss: 1.0633
Epoch 7/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9974 - loss: 0.0159 - val_accuracy: 0.7126 - val_loss: 1.1366
Epoch 8/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9990 - loss: 0.0099 - val_accuracy: 0.7294 - val_loss: 1.3207


In [7]:
mlp_loss, mlp_acc = mlp_model.evaluate(X_test_pad, y_test_enc, verbose=0)
results = {}
results['MLP'] = {'test_accuracy': round(mlp_acc, 4), 'test_loss': round(mlp_loss, 4)}
print(f'MLP | 정확도: {mlp_acc:.4f} | 손실: {mlp_loss:.4f}')

MLP | 정확도: 0.6814 | 손실: 0.7472


### 5-2. CNN

In [8]:
early_stop_cnn = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

cnn_model = Sequential([
    Embedding(MAX_WORDS, EMBED_DIM),
    Conv1D(64, kernel_size=3, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')
])

cnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history_cnn = cnn_model.fit(
    X_train_pad, y_train_enc,
    epochs=50, batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop_cnn],
    verbose=1
)

Epoch 1/50


I0000 00:00:1775544384.811031    1839 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_8336__.22
I0000 00:00:1775544385.729015    1839 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
I0000 00:00:1775544386.092945    2740 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_8', 12 bytes spill stores, 12 bytes spill loads

I0000 00:00:1775544386.928551    1839 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
I0000 00:00:1775544387.305918    2744 subprocess_compilation.cc:348] ptxas warning : Registers are spilled 

83/97 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5890 - loss: 0.9833

I0000 00:00:1775544388.578415    1839 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_8336__.22
I0000 00:00:1775544389.009950    1839 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
I0000 00:00:1775544389.363205    2874 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_8', 12 bytes spill stores, 12 bytes spill loads

I0000 00:00:1775544389.404533    1839 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
I0000 00:00:1775544389.725754    2872 subprocess_compilation.cc:348] ptxas warning : Registers are spilled 

97/97 ━━━━━━━━━━━━━━━━━━━━ 9s 40ms/step - accuracy: 0.5903 - loss: 0.9402 - val_accuracy: 0.6070 - val_loss: 0.8808
Epoch 2/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6387 - loss: 0.8168 - val_accuracy: 0.6894 - val_loss: 0.7296
Epoch 3/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7681 - loss: 0.5703 - val_accuracy: 0.7448 - val_loss: 0.6252
Epoch 4/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8942 - loss: 0.3098 - val_accuracy: 0.7668 - val_loss: 0.6172
Epoch 5/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9552 - loss: 0.1521 - val_accuracy: 0.7719 - val_loss: 0.6859
Epoch 6/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9842 - loss: 0.0734 - val_accuracy: 0.7564 - val_loss: 0.8221
Epoch 7/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9926 - loss: 0.0377 - val_accuracy: 0.7461 - val_loss: 0.8222
Epoch 8/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9974 - loss: 0.0213 - val_accuracy: 0.7487 - val_loss: 0.9127
Ep

In [9]:
cnn_loss, cnn_acc = cnn_model.evaluate(X_test_pad, y_test_enc, verbose=0)
results['CNN'] = {'test_accuracy': round(cnn_acc, 4), 'test_loss': round(cnn_loss, 4)}
print(f"MLP | 정확도: {results['MLP']['test_accuracy']} | 손실: {results['MLP']['test_loss']}")
print(f'CNN | 정확도: {cnn_acc:.4f} | 손실: {cnn_loss:.4f}')

MLP | 정확도: 0.6814 | 손실: 0.7472
CNN | 정확도: 0.7515 | 손실: 0.7129


### 5-3. LSTM

- 클래스 불균형(neutral 59%)으로 인해 다수 클래스만 예측하는 현상 발생 가능
- 데이터 규모(4,846건)가 작아 RNN 계열이 학습에 어려움을 겪을 수 있음

In [10]:
early_stop_lstm = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

lstm_model = Sequential([
    Embedding(MAX_WORDS, EMBED_DIM),
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

lstm_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history_lstm = lstm_model.fit(
    X_train_pad, y_train_enc,
    epochs=50, batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop_lstm],
    verbose=1
)

Epoch 1/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - accuracy: 0.5842 - loss: 0.9550 - val_accuracy: 0.6070 - val_loss: 0.9207
Epoch 2/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.5910 - loss: 0.9361 - val_accuracy: 0.6070 - val_loss: 0.9172
Epoch 3/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.5910 - loss: 0.9336 - val_accuracy: 0.6070 - val_loss: 0.9180
Epoch 4/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.5910 - loss: 0.9296 - val_accuracy: 0.6070 - val_loss: 0.9211
Epoch 5/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.5910 - loss: 0.9317 - val_accuracy: 0.6070 - val_loss: 0.9176
Epoch 6/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.5910 - loss: 0.9303 - val_accuracy: 0.6070 - val_loss: 0.9187
Epoch 7/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.5910 - loss: 0.9328 - val_accuracy: 0.6070 - val_loss: 0.9191


In [11]:
lstm_loss, lstm_acc = lstm_model.evaluate(X_test_pad, y_test_enc, verbose=0)
results['LSTM'] = {'test_accuracy': round(lstm_acc, 4), 'test_loss': round(lstm_loss, 4)}
print(f"MLP  | 정확도: {results['MLP']['test_accuracy']} | 손실: {results['MLP']['test_loss']}")
print(f"CNN  | 정확도: {results['CNN']['test_accuracy']} | 손실: {results['CNN']['test_loss']}")
print(f'LSTM | 정확도: {lstm_acc:.4f} | 손실: {lstm_loss:.4f}')

MLP  | 정확도: 0.6814 | 손실: 0.7472
CNN  | 정확도: 0.7515 | 손실: 0.7129
LSTM | 정확도: 0.5938 | 손실: 0.9261


### 5-4. GRU

In [12]:
early_stop_gru = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

gru_model = Sequential([
    Embedding(MAX_WORDS, EMBED_DIM),
    GRU(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

gru_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history_gru = gru_model.fit(
    X_train_pad, y_train_enc,
    epochs=50, batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop_gru],
    verbose=1
)

Epoch 1/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.5900 - loss: 0.9432 - val_accuracy: 0.6070 - val_loss: 0.9199
Epoch 2/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.5910 - loss: 0.9345 - val_accuracy: 0.6070 - val_loss: 0.9193
Epoch 3/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.5910 - loss: 0.9307 - val_accuracy: 0.6070 - val_loss: 0.9206
Epoch 4/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.5910 - loss: 0.9328 - val_accuracy: 0.6070 - val_loss: 0.9189
Epoch 5/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.5910 - loss: 0.9291 - val_accuracy: 0.6070 - val_loss: 0.9209
Epoch 6/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.5910 - loss: 0.9302 - val_accuracy: 0.6070 - val_loss: 0.9215
Epoch 7/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.5910 - loss: 0.9293 - val_accuracy: 0.6070 - val_loss: 0.9188
Epoch 8/50
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.5910 - loss: 0.9320 - val_accuracy: 0.6070 - va

In [13]:
gru_loss, gru_acc = gru_model.evaluate(X_test_pad, y_test_enc, verbose=0)
results['GRU'] = {'test_accuracy': round(gru_acc, 4), 'test_loss': round(gru_loss, 4)}
print(f"MLP  | 정확도: {results['MLP']['test_accuracy']} | 손실: {results['MLP']['test_loss']}")
print(f"CNN  | 정확도: {results['CNN']['test_accuracy']} | 손실: {results['CNN']['test_loss']}")
print(f"LSTM | 정확도: {results['LSTM']['test_accuracy']} | 손실: {results['LSTM']['test_loss']}")
print(f'GRU  | 정확도: {gru_acc:.4f} | 손실: {gru_loss:.4f}')

MLP  | 정확도: 0.6814 | 손실: 0.7472
CNN  | 정확도: 0.7515 | 손실: 0.7129
LSTM | 정확도: 0.5938 | 손실: 0.9261
GRU  | 정확도: 0.5938 | 손실: 0.9261


## 6. 최종 결과 비교

In [14]:
print('===== 최종 결과 비교 =====')
best_model = max(results, key=lambda x: results[x]['test_accuracy'])
for name, r in results.items():
    marker = ' <- 최고 성능' if name == best_model else ''
    print(f"{name:>5} | 정확도: {r['test_accuracy']} | 손실: {r['test_loss']}{marker}")

===== 최종 결과 비교 =====
  MLP | 정확도: 0.6814 | 손실: 0.7472
  CNN | 정확도: 0.7515 | 손실: 0.7129 <- 최고 성능
 LSTM | 정확도: 0.5938 | 손실: 0.9261
  GRU | 정확도: 0.5938 | 손실: 0.9261


## 7. 실제 금융 뉴스 예측

Yahoo Finance 뉴스 기사를 크롤링해서 감성 분석

In [15]:
def get_prediction(text):
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')
    pred = cnn_model.predict(pad, verbose=0)[0]
    return le.classes_[np.argmax(pred)], pred


def predict_sentiment_pretty(texts, titles=None):
    emoji = {'positive': '🟢', 'neutral': '🟡', 'negative': '🔴'}
    for i, text in enumerate(texts):
        _, pred = get_prediction(text)
        result = le.classes_[np.argmax(pred)]
        print('=' * 60)
        if titles:
            print(f'제목: {titles[i]}')
        print(f'예측: {emoji[result]} {result.upper()}')
        print(f'  negative: {pred[0]:.2%}')
        print(f'  neutral:  {pred[1]:.2%}')
        print(f'  positive: {pred[2]:.2%}')
        print()

In [16]:
from newspaper import Article
from nltk.tokenize import sent_tokenize

urls = [
    'https://finance.yahoo.com/sectors/energy/articles/no-other-choice-war-hit-031518842.html',
    'https://finance.yahoo.com/news/live/stock-market-today-dow-soars-1000-points-sp-500-and-nasdaq-surge-in-upbeat-end-to-brutal-quarter-184154822.html',
    'https://finance.yahoo.com/news/oil-plunges-stocks-soar-after-iranian-president-offers-first-signs-of-willingness-to-end-war-with-us-183158792.html',
    'https://finance.yahoo.com/news/airlines-face-price-hikes-lower-margins-as-iran-war-pressures-business-171745222.html'
]

full_texts = []
titles = []
for url in urls:
    try:
        article = Article(url)
        article.download()
        article.parse()
        full_texts.append(article.text)
        titles.append(article.title)
        print(f'완료: {article.title}')
    except Exception as e:
        print(f'실패: {url}')

완료: ‘There’s No Other Choice’: War-Hit Asian Buyers Grab Russian Oil
완료: Stock market today: Dow soars 1,000 points, S&P 500 and Nasdaq surge in upbeat end to brutal quarter
완료: Oil plunges, stocks soar after Iranian president offers first signs of willingness to end war with US
완료: Airlines face price hikes, lower margins as Iran war pressures business


### 7-1. 본문 전체 예측

In [17]:
predict_sentiment_pretty(full_texts, titles=titles)

제목: ‘There’s No Other Choice’: War-Hit Asian Buyers Grab Russian Oil
예측: 🟡 NEUTRAL
  negative: 1.70%
  neutral:  82.69%
  positive: 15.61%

제목: Stock market today: Dow soars 1,000 points, S&P 500 and Nasdaq surge in upbeat end to brutal quarter
예측: 🟢 POSITIVE
  negative: 2.78%
  neutral:  28.64%
  positive: 68.58%

제목: Oil plunges, stocks soar after Iranian president offers first signs of willingness to end war with US
예측: 🟡 NEUTRAL
  negative: 2.58%
  neutral:  85.57%
  positive: 11.84%

제목: Airlines face price hikes, lower margins as Iran war pressures business
예측: 🟡 NEUTRAL
  negative: 2.67%
  neutral:  84.58%
  positive: 12.76%



### 7-2. 제목 + 앞 3문장 예측

In [18]:
summary_texts = [' '.join(sent_tokenize(text)[:3]) for text in full_texts]
predict_sentiment_pretty(summary_texts, titles=titles)

제목: ‘There’s No Other Choice’: War-Hit Asian Buyers Grab Russian Oil
예측: 🟡 NEUTRAL
  negative: 1.53%
  neutral:  82.58%
  positive: 15.89%

제목: Stock market today: Dow soars 1,000 points, S&P 500 and Nasdaq surge in upbeat end to brutal quarter
예측: 🟢 POSITIVE
  negative: 2.78%
  neutral:  28.64%
  positive: 68.58%

제목: Oil plunges, stocks soar after Iranian president offers first signs of willingness to end war with US
예측: 🟡 NEUTRAL
  negative: 4.47%
  neutral:  78.90%
  positive: 16.63%

제목: Airlines face price hikes, lower margins as Iran war pressures business
예측: 🟡 NEUTRAL
  negative: 0.78%
  neutral:  94.84%
  positive: 4.38%



### 7-3. 비교표 (본문 전체 vs 앞 3문장)

In [19]:
def get_pred_str(text):
    result, pred = get_prediction(text)
    return f'{result}({pred[np.argmax(pred)]:.0%})'

print(f"{'제목':<45} {'본문전체':^20} {'앞3문장':^20}")
print('=' * 85)
for i in range(len(titles)):
    title = titles[i][:40]
    print(f'{title:<45} {get_pred_str(full_texts[i]):^20} {get_pred_str(summary_texts[i]):^20}')

제목                                                    본문전체                 앞3문장        
‘There’s No Other Choice’: War-Hit Asian          neutral(83%)         neutral(83%)    
Stock market today: Dow soars 1,000 poin         positive(69%)        positive(69%)    
Oil plunges, stocks soar after Iranian p          neutral(86%)         neutral(79%)    
Airlines face price hikes, lower margins          neutral(85%)         neutral(95%)    


## 8. URL 분석 함수

URL을 넣으면 본문 전체 / 제목+앞3문장 두 가지로 자동 분석

In [20]:
def analyze_url(url):
    """URL 하나를 받아서 본문 전체 / 제목+앞3문장 두 가지로 감성 분석"""
    try:
        article = Article(url)
        article.download()
        article.parse()
    except Exception as e:
        print(f'기사 로드 실패: {e}')
        return

    title = article.title
    full_text = article.text
    summary_text = ' '.join(sent_tokenize(full_text)[:3])
    result_full, pred_full = get_prediction(full_text)
    result_summary, pred_summary = get_prediction(summary_text)
    emoji = {'positive': '🟢', 'neutral': '🟡', 'negative': '🔴'}

    print('=' * 60)
    print(f'제목: {title}')
    print()
    print('[본문 전체]')
    print(f'  예측: {emoji[result_full]} {result_full.upper()}')
    print(f'  negative: {pred_full[0]:.2%} | neutral: {pred_full[1]:.2%} | positive: {pred_full[2]:.2%}')
    print()
    print('[제목 + 앞 3문장]')
    print(f'  예측: {emoji[result_summary]} {result_summary.upper()}')
    print(f'  negative: {pred_summary[0]:.2%} | neutral: {pred_summary[1]:.2%} | positive: {pred_summary[2]:.2%}')
    print('=' * 60)


def analyze_urls(urls):
    """URL 여러 개를 받아서 분석"""
    for url in urls:
        analyze_url(url)
        print()


# 사용 예시
# analyze_url('https://finance.yahoo.com/news/...')
# analyze_urls(['URL1', 'URL2', 'URL3'])